In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv(r'C:billing_and_costs_m.csv')

In [83]:
print(df.shape)
print(df.dtypes)

(13429, 12)
billing_id                  str
admission_id                str
patient_id                  str
billing_date                str
total_treatment_cost    float64
medication_cost         float64
lab_cost                float64
procedure_cost          float64
revenue_collected       float64
insurance_payout        float64
out_of_pocket           float64
payment_status              str
dtype: object


In [84]:
df.head()

,billing_id,admission_id,patient_id,billing_date,total_treatment_cost,medication_cost,lab_cost,procedure_cost,revenue_collected,insurance_payout,out_of_pocket,payment_status
0,BIL-000001,ADM-002070,PAT-07844,1/3/2021,2415.02,539.51,376.28,917.53,1710.32,1353.91,356.41,Paid
1,BIL-000002,ADM-009484,PAT-06074,7/1/2019,4268.26,1474.63,995.55,1203.01,3104.19,2131.59,972.60,Paid
2,BIL-000003,ADM-006331,PAT-01784,11/10/2022,500.00,121.84,105.59,135.46,326.29,236.00,90.29,PAID
3,BIL-000004,ADM-001084,PAT-07425,2/6/2017,5622.23,1786.61,955.60,1839.23,4882.81,3219.19,1663.62,paid
4,BIL-000005,ADM-009785,PAT-00566,12/16/2019,3560.19,764.20,777.63,1125.10,2757.04,2065.12,691.92,Paid


In [85]:
print(df['total_treatment_cost'].isna().sum())
print((df['total_treatment_cost'] < 0).sum())

1074
268


In [86]:
# Filter rows with negative values
df[df['total_treatment_cost'] < 0].head()

,billing_id,admission_id,patient_id,billing_date,total_treatment_cost,medication_cost,lab_cost,procedure_cost,revenue_collected,insurance_payout,out_of_pocket,payment_status
52,BIL-000053,ADM-005675,PAT-05872,2/17/2022,-789.07,671.69,677.70,1062.31,2855.19,2275.63,579.56,Paid
115,BIL-000116,ADM-009388,PAT-00081,9/25/2022,-5782.05,1832.65,1437.60,2737.61,5780.84,4158.68,1622.16,Paid
117,BIL-000118,ADM-001311,PAT-08090,4/24/2020,-4365.96,1707.99,1397.09,2315.55,4044.74,3165.85,878.89,written-off
123,BIL-000124,ADM-003596,PAT-02045,8/13/2021,-5503.96,1477.40,1575.90,1896.51,5467.29,3814.18,1653.11,Paid
129,BIL-000130,ADM-008285,PAT-04889,12/31/2023,-1727.37,986.13,677.31,1550.32,3912.09,2954.17,957.92,PAID


In [87]:
# remove negative values 
df['total_treatment_cost'] = df['total_treatment_cost'].abs()

In [88]:
# Add new claculated column 
df['cost_sum'] = df['medication_cost'] + df['lab_cost'] + df['procedure_cost']

In [89]:
broken = df[df['total_treatment_cost'] < df['cost_sum']]

In [90]:
print(broken.shape)
broken.head()

(230, 13)


,billing_id,admission_id,patient_id,billing_date,total_treatment_cost,medication_cost,lab_cost,procedure_cost,revenue_collected,insurance_payout,out_of_pocket,payment_status,cost_sum
52,BIL-000053,ADM-005675,PAT-05872,2/17/2022,789.07,671.69,677.70,1062.31,2855.19,2275.63,579.56,Paid,2411.70
115,BIL-000116,ADM-009388,PAT-00081,9/25/2022,5782.05,1832.65,1437.60,2737.61,5780.84,4158.68,1622.16,Paid,6007.86
117,BIL-000118,ADM-001311,PAT-08090,4/24/2020,4365.96,1707.99,1397.09,2315.55,4044.74,3165.85,878.89,written-off,5420.63
129,BIL-000130,ADM-008285,PAT-04889,12/31/2023,1727.37,986.13,677.31,1550.32,3912.09,2954.17,957.92,PAID,3213.76
157,BIL-000158,ADM-010912,PAT-03458,12/19/2019,2124.65,885.18,612.92,930.46,2207.89,1818.11,389.78,paid,2428.56


In [91]:
# Fix corrupt data entry in total_treatment_cost
df.loc[df['total_treatment_cost'] < df['cost_sum'], 'total_treatment_cost'] = df['cost_sum']
print(broken.shape)

(230, 13)


In [92]:
# Convert string to datetime datatype
df['billing_date'] = pd.to_datetime(df['billing_date'])
print(df.dtypes)

billing_id                         str
admission_id                       str
patient_id                         str
billing_date            datetime64[us]
total_treatment_cost           float64
medication_cost                float64
lab_cost                       float64
procedure_cost                 float64
revenue_collected              float64
insurance_payout               float64
out_of_pocket                  float64
payment_status                     str
cost_sum                       float64
dtype: object


In [93]:
# Fix data inconsistency in payment_status
df['payment_status'] = df['payment_status'].str.strip().str.title()
print(df['payment_status'].value_counts())

payment_status
Paid           9052
Pending        2974
Written-Off    1403
Name: count, dtype: int64


In [94]:
# Analyze if revenue exceeds total_treatment_cost
df[df['revenue_collected'] > df['total_treatment_cost']].shape

(776, 13)

In [95]:
prob = df[df['revenue_collected'] > df['total_treatment_cost']].copy()

# How big is the exces amaount?
prob['exces'] = prob['revenue_collected'] - prob['total_treatment_cost']

print(prob[['billing_id', 'total_treatment_cost', 'revenue_collected', 'exces']].head(10))
print()
print(prob['exces'].describe())

    billing_id  total_treatment_cost  revenue_collected    exces
11  BIL-000012               4836.01            6964.14  2128.13
15  BIL-000016               5054.47            5361.41   306.94
24  BIL-000025               5561.25            6753.90  1192.65
39  BIL-000040               3010.81            4086.45  1075.64
47  BIL-000048               6648.02            7674.89  1026.87
51  BIL-000052               2942.34            3796.82   854.48
52  BIL-000053               2411.70            2855.19   443.49
56  BIL-000057               3612.40            4896.21  1283.81
59  BIL-000060               3872.09            5065.62  1193.53
66  BIL-000067              16120.31           21687.42  5567.11

count     776.000000
mean     1153.633003
std      1035.121504
min         7.330000
25%       423.217500
50%       832.965000
75%      1600.620000
max      7063.550000
Name: exces, dtype: float64


In [96]:
# Assign revenue_collected col to total_treatment_cost
mask = df['revenue_collected'] > df['total_treatment_cost']
df.loc[mask, 'total_treatment_cost'] = df['revenue_collected']

print(df[df['revenue_collected'] > df['total_treatment_cost']].shape)

(0, 13)


In [103]:
df.head()

,billing_id,admission_id,patient_id,billing_date,total_treatment_cost,medication_cost,lab_cost,procedure_cost,revenue_collected,insurance_payout,out_of_pocket,payment_status
0,BIL-000001,ADM-002070,PAT-07844,2021-01-03,2415.02,539.51,376.28,917.53,1710.32,1353.91,356.41,Paid
1,BIL-000002,ADM-009484,PAT-06074,2019-07-01,4268.26,1474.63,995.55,1203.01,3104.19,2131.59,972.60,Paid
2,BIL-000003,ADM-006331,PAT-01784,2022-11-10,500.00,121.84,105.59,135.46,326.29,236.00,90.29,Paid
3,BIL-000004,ADM-001084,PAT-07425,2017-02-06,5622.23,1786.61,955.60,1839.23,4882.81,3219.19,1663.62,Paid
4,BIL-000005,ADM-009785,PAT-00566,2019-12-16,3560.19,764.20,777.63,1125.10,2757.04,2065.12,691.92,Paid


In [104]:
df.to_csv('billing_and_costs_c', index=False)